<a href="https://colab.research.google.com/github/JakeBusler/ColabVectorizer/blob/main/Vectorizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title **Prepare the environment**

# Upgrade pip, setuptools, and wheel to prevent build issues
!pip install -q --upgrade pip setuptools wheel

# Install missing dependency for ipython
!pip install -q jedi

# Install system-level build tools
!apt-get update -qq
!apt-get install -y -qq build-essential python3-dev

# Install Hugging Face ecosystem and Gradio
!pip install -q transformers accelerate bitsandbytes gradio pillow

# Install the custom starvector module straight from the source repository
!pip install -q git+https://github.com/joanrod/star-vector.git

# Fetch and install the official cloudflared binary directly from the source
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb








In [ ]:
# @title **Import the model and launch the server**

import gradio as gr
import torch
import subprocess
import time
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
from PIL import Image

# 1. Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "starvector/starvector-8b-im2svg"

# 2. Load processor and model with trust_remote_code explicitly set to True
print("Loading processor and model (this may take a few minutes)...")
processor = AutoProcessor.from_pretrained(
    model_id,
    trust_remote_code=True
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quantization_config,
    trust_remote_code=True
)

# 3. Define the Inference Function
def vectorize_image(image):
    if image is None:
        return "No image provided."

    image = image.convert("RGB")
    prompt = "<image>\nConvert this image to SVG code:"

    inputs = processor(text=prompt, images=image, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=2048)

    decoded = processor.batch_decode(outputs, skip_special_tokens=True)[0]

    if "<svg" in decoded:
        svg_code = decoded[decoded.find("<svg"):]
    else:
        svg_code = decoded

    return svg_code

# 4. Create the Gradio Interface
interface = gr.Interface(
    fn=vectorize_image,
    inputs=gr.Image(type="pil", label="Upload Raster Image"),
    outputs=gr.Code(language="html", label="Generated SVG Code"),
    title="StarVector 8B Image to SVG",
    description="Upload an image to generate SVG code. Running on 4-bit quantization."
)

# 5. Launch Gradio and Cloudflare Tunnel
print("Launching Gradio locally...")
interface.launch(server_port=7860, inline=False, prevent_thread_lock=True)

print("Establishing Cloudflare Tunnel...")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in tunnel.stdout:
    if "trycloudflare.com" in line:
        url = [word for word in line.split() if "trycloudflare.com" in word][0]
        print("\n" + "="*50)
        print(f"PUBLIC ACCESS URL: {url}")
        print("="*50 + "\n")
        break

while True:
    time.sleep(1)